In [1]:
import subprocess, sys
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True)
print(result.stdout.strip())
import torch
assert torch.cuda.is_available(), "No GPU detected"
props = torch.cuda.get_device_properties(0)
assert props.total_memory > 30e9, (
    f"Need ≥40 GB VRAM. Got {props.total_memory/1e9:.0f} GB. Switch to A100.")
print(f"GPU: {props.name} — {props.total_memory/1e9:.0f} GB")

NVIDIA A100-SXM4-80GB, 81920
GPU: NVIDIA A100-SXM4-80GB — 85 GB


In [2]:
!pip install flask pyngrok nilearn nibabel -q
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.26.4,<2.1.0'])

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchvision==0.25.0',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu124',
                '--no-deps'])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git'])

print("So far good man")

So far good man


In [3]:
import subprocess, sys

# Base deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'flask', 'flask-cors', 'pyngrok', 'nilearn', 'nibabel'])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git'])


subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                'numpy==2.0.2'])

import numpy
print("numpy:", numpy.__version__)
print("\n⚠️  Runtime → Restart session NOW, then run the model load cell")
print("    (numpy is already imported in this kernel — restart is mandatory)")

numpy: 2.0.2

⚠️  Runtime → Restart session NOW, then run the model load cell
    (numpy is already imported in this kernel — restart is mandatory)


In [4]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    from huggingface_hub import login
    login()
    print("HF_TOKEN loaded from huggingface_hub")
from pathlib import Path
from tribev2.demo_utils import TribeModel
import torch
CACHE_DIR = Path('/content/tribe_cache')
CACHE_DIR.mkdir(exist_ok=True)
print('Loading TRIBE v2 (first run downloads ~1 GB)...')
model = TribeModel.from_pretrained(
    'facebook/tribev2',
    cache_folder=str(CACHE_DIR)
)
print('Model loaded')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM after load: {used:.1f} / {total:.1f} GB')

HF_TOKEN loaded from Colab Secrets


/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-20 12:21:38 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Loading TRIBE v2 (first run downloads ~1 GB)...


2026-06-20 12:21:39 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

Model loaded
VRAM after load: 0.7 / 85.1 GB


In [5]:
import numpy as np
from nilearn import datasets as nl_datasets
from nilearn.plotting import view_surf
from IPython.display import display, HTML
N_PER_HEMI = 10242
print('Fetching fsaverage5 mesh...')
fsavg = nl_datasets.fetch_surf_fsaverage(mesh='fsaverage5')
print('Get Ready for the Mesh')
print('Keys:', [k for k in fsavg.keys() if k != 'description'])

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7c820914e660>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


Fetching fsaverage5 mesh...
Get Ready for the Mesh
Keys: ['area_left', 'area_right', 'curv_left', 'curv_right', 'flat_left', 'flat_right', 'infl_left', 'infl_right', 'pial_left', 'pial_right', 'sphere_left', 'sphere_right', 'sulc_left', 'sulc_right', 'thick_left', 'thick_right', 'white_left', 'white_right']


In [6]:
import numpy as np
import nibabel as nib
import urllib.request, os
from nilearn.surface import load_surf_mesh

os.makedirs('/content/tribe_cache', exist_ok=True)
print("Downloading HCP-MMP1 atlas...")
atlas_path = '/content/tribe_cache/MMP_in_MNI_corr.nii.gz'
if not os.path.exists(atlas_path):
    urllib.request.urlretrieve(
        'https://ndownloader.figshare.com/files/5594363', atlas_path)

atlas_img  = nib.load(atlas_path)
atlas_data = np.asarray(atlas_img.get_fdata(), dtype=np.int32)
inv_affine = np.linalg.inv(atlas_img.affine)

lh_coords  = load_surf_mesh(fsavg['pial_left'])[0]
rh_coords  = load_surf_mesh(fsavg['pial_right'])[0]
all_coords = np.vstack([lh_coords, rh_coords])

coords_h = np.hstack([all_coords, np.ones((len(all_coords), 1))])
vox_ijk  = np.round((inv_affine @ coords_h.T).T[:, :3]).astype(int)
nx, ny, nz = atlas_data.shape
valid = (
    (vox_ijk[:,0] >= 0) & (vox_ijk[:,0] < nx) &
    (vox_ijk[:,1] >= 0) & (vox_ijk[:,1] < ny) &
    (vox_ijk[:,2] >= 0) & (vox_ijk[:,2] < nz)
)
vertex_labels = np.zeros(len(all_coords), dtype=np.int32)
vi = vox_ijk[valid]
vertex_labels[valid] = atlas_data[vi[:,0], vi[:,1], vi[:,2]]

GLASSER = {
    "V4": 6, "FFC": 18, "MT": 23, "MST": 2, "V4t": 156, "FST": 157,
    "a24": 61, "p24": 179, "p32": 63, "s32": 64, "24dd": 40, "24dv": 41,
    "46": 84, "9-46d": 86, "p9-46v": 83, "a9-46v": 85,
    "AAIC": 112, "AVI": 111, "MI": 109, "PI": 110,
}
REGION_MAP = {
    "FFA":    ("FFC",),
    "V4":     ("V4",),
    "MT+":    ("MT", "MST", "V4t", "FST"),
    "PFC":    ("46", "9-46d", "p9-46v", "a9-46v"),
    "ACC":    ("a24", "p24", "p32", "s32", "24dd", "24dv"),
    "Insula": ("AAIC", "AVI", "MI", "PI"),
}

region_masks = {}
for region, names in REGION_MAP.items():
    idx = [i for n in names for lh in [GLASSER.get(n)] if lh for i in (lh, lh+180)]
    if idx:
        region_masks[region] = np.isin(vertex_labels, idx)
        print(f"  {region:8s}: {int(region_masks[region].sum())} vertices")

  FFA     : 25 vertices
  V4      : 35 vertices
  MT+     : 172 vertices
  PFC     : 310 vertices
  ACC     : 360 vertices
  Insula  : 264 vertices


In [7]:
import nibabel as nib
import numpy as np
from nilearn import datasets as nl_datasets

# fetch_atlas_harvard_oxford returns the image directly in newer nilearn
atlas = nl_datasets.fetch_atlas_harvard_oxford('sub-maxprob-thr25-2mm')

# Handle both old and new nilearn versions
if hasattr(atlas.maps, 'get_fdata'):
    # Newer nilearn — atlas.maps is already a Nifti1Image
    sub_img  = atlas.maps
else:
    # Older nilearn — atlas.maps is a file path
    sub_img  = nib.load(atlas.maps)

sub_data   = np.asarray(sub_img.get_fdata(), dtype=np.int32)
sub_affine = sub_img.affine

SUBCORTICAL_LABELS = {
    'Hippocampus': [6, 13],
    'Amygdala':    [7, 14],
    'NAcc':        [8, 15],
}

subcortical_masks = {}
for region, label_ids in SUBCORTICAL_LABELS.items():
    mask       = np.isin(sub_data, label_ids)
    vox_coords = np.argwhere(mask)
    mni_coords = nib.affines.apply_affine(sub_affine, vox_coords)
    subcortical_masks[region] = mni_coords
    print(f"  {region:15s}: {len(mni_coords)} voxels")

print("✓ Subcortical masks ready")

[fetch_atlas_harvard_oxford] Dataset found in /root/nilearn_data/fsl

  Hippocampus    : 64853 voxels
  Amygdala       : 1286 voxels
  NAcc           : 6173 voxels
✓ Subcortical masks ready


In [8]:
import cv2, io, os, tempfile
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial import cKDTree
from nilearn.surface import load_surf_mesh
from tribev2.demo_utils import get_audio_and_text_events

def tribe_predict_image(img_bytes: bytes) -> dict:
    img    = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_np = np.array(img)
    h, w   = img_np.shape[:2]
    w, h   = w - w % 2, h - h % 2
    img_np = img_np[:h, :w]

    with tempfile.TemporaryDirectory() as tmpdir:
        video_path = os.path.join(tmpdir, 'ui.mp4')
        writer = cv2.VideoWriter(
            video_path, cv2.VideoWriter_fourcc(*'mp4v'), 1, (w, h))
        for _ in range(2):
            writer.write(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
        writer.release()

        event_df = pd.DataFrame([{
            "type":     "Video",
            "filepath": video_path,
            "start":    0,
            "timeline": "default",
            "subject":  "default",
        }])
        events   = get_audio_and_text_events(event_df)
        preds, _ = model.predict(events=events)

    preds_np = np.asarray(preds)
    mean_act = preds_np.mean(axis=0)

    # Cortical region scores
    scores = {
        r: float(mean_act[m].mean()) if m.sum() > 0 else 0.0
        for r, m in region_masks.items()
    }
    lo, hi = min(scores.values()), max(scores.values())
    if hi - lo > 1e-6:
        scores = {k: (v - lo) / (hi - lo) for k, v in scores.items()}

    # Subcortical estimates via nearest cortical vertex
    lh_coords  = load_surf_mesh(fsavg['pial_left'])[0]
    rh_coords  = load_surf_mesh(fsavg['pial_right'])[0]
    all_coords = np.vstack([lh_coords, rh_coords])
    tree       = cKDTree(all_coords)

    subcortical_scores = {}
    for region, mni_coords in subcortical_masks.items():
        _, idx = tree.query(mni_coords)
        subcortical_scores[region] = float(mean_act[idx].mean())

    lo2, hi2 = min(subcortical_scores.values()), max(subcortical_scores.values())
    if hi2 - lo2 > 1e-6:
        subcortical_scores = {k: (v - lo2) / (hi2 - lo2)
                              for k, v in subcortical_scores.items()}

    return {
        "vertices": {
            "lh": mean_act[:10242].tolist(),
            "rh": mean_act[10242:].tolist(),
        },
        "region_scores": scores,
        "subcortical_estimates": {
            "values": subcortical_scores,
            "method": "nearest_cortical_vertex",
            "warning": "Cortical proxy estimates only. TRIBE v2 models cortical activity only."
        },
        "meta": {
            "n_vertices_per_hemi": 10242,
            "n_timesteps": int(preds_np.shape[0]),
            "surface": "fsaverage5",
        }
    }

In [9]:
checks = {
    'model':               'model' in globals(),
    'region_masks':        'region_masks' in globals(),
    'subcortical_masks':   'subcortical_masks' in globals(),
    'tribe_predict_image': 'tribe_predict_image' in globals(),
    'fsavg':               'fsavg' in globals(),
}
for name, ok in checks.items():
    print(f"{'✓' if ok else '✗'} {name}")
assert all(checks.values()), "Fix ✗ items before starting server"
print("\n✓ All good — safe to start server")

✓ model
✓ region_masks
✓ subcortical_masks
✓ tribe_predict_image
✓ fsavg

✓ All good — safe to start server


In [10]:
import subprocess, torch

# GPU memory
free_gpu, total_gpu = torch.cuda.mem_get_info()
print(f"GPU: {free_gpu/1e9:.1f} GB free / {total_gpu/1e9:.1f} GB total")

# System RAM
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result.stdout)

# Check if anything is already using the GPU
result2 = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result2.stdout)

import subprocess
result = subprocess.run(['fuser', '8080/tcp'], capture_output=True, text=True)
print("Port 8080 in use by PID:", result.stdout.strip() or "nobody")

# Flask
import flask
print("Flask version:", flask.__version__)

GPU: 83.9 GB free / 85.1 GB total
               total        used        free      shared  buff/cache   available
Mem:           167Gi       2.1Gi       121Gi        29Mi        43Gi       163Gi
Swap:             0B          0B          0B

Sat Jun 20 12:21:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |              

/tmp/ipykernel_5780/4052475714.py:21: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [11]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok
import os
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

srv = Flask(__name__)

@srv.route('/encode', methods=['POST'])
def encode():
    if not request.data:
        return jsonify({'error': 'POST raw PNG bytes as request body'}), 400
    return jsonify(tribe_predict_image(request.data))

@srv.route('/health')
def health():
    return jsonify({'ok': True})

PORT = 5000
os.system(f"fuser -k {PORT}/tcp")
threading.Thread(target=lambda: srv.run(port=PORT, use_reloader=False), daemon=True).start()
tunnel = ngrok.connect(PORT)
print("TRIBE endpoint:", tunnel.public_url + "/encode")
print("Health check: ", tunnel.public_url + "/health")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


TRIBE endpoint: https://00ac-34-143-253-73.ngrok-free.app/encode
Health check:  https://00ac-34-143-253-73.ngrok-free.app/health


In [12]:
import requests
print(requests.get(tunnel.public_url + "/health").json())

INFO:werkzeug:127.0.0.1 - - [20/Jun/2026 12:22:03] "GET /health HTTP/1.1" 200 -


{'ok': True}


In [1]:
!pip install flask-cors -q

import threading, os
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata

# Kill old server + tunnels
os.system("fuser -k 5000/tcp")
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

srv = Flask(__name__)
CORS(srv)  # allow browser clients

@srv.route('/encode', methods=['POST'])
def encode():
    if not request.data:
        return jsonify({'error': 'POST raw PNG bytes as request body'}), 400
    try:
        return jsonify(tribe_predict_image(request.data))
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@srv.route('/health')
def health():
    return jsonify({'ok': True})

PORT = 5000
threading.Thread(
    target=lambda: srv.run(port=PORT, threaded=True, use_reloader=False),
    daemon=True
).start()

tunnel = ngrok.connect(PORT)
URL = tunnel.public_url

print("=" * 60)
print("TRIBE v2 endpoint live")
print("=" * 60)
print(f"POST   {URL}/encode")
print(f"GET    {URL}/health")
print("=" * 60)
print("\nShare message ↓↓↓ copy everything below this line\n")
print(f"""TRIBE v2 brain-grounded encoder — live endpoint
POST a PNG → get fsaverage5 cortical activations + region scores
(FFA, V4, MT+, PFC, ACC, Insula) + subcortical proxies.

Endpoint: {URL}/encode
Health:   {URL}/health

Python:
import requests
with open('thumb.png','rb') as f:
    r = requests.post('{URL}/encode', data=f.read(), timeout=120)
print(r.json()['region_scores'])

curl:
curl -X POST --data-binary @thumb.png {URL}/encode

Notes:
- Hosted on my Colab — ping me if it 502s
- ~40 req/min cap (ngrok free), ~20s first call (warm-up)
- Subcortical = cortical-proxy estimates, not raw subcortical signal
""")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


TRIBE v2 endpoint live
POST   https://b227-34-143-253-73.ngrok-free.app/encode
GET    https://b227-34-143-253-73.ngrok-free.app/health

Share message ↓↓↓ copy everything below this line

TRIBE v2 brain-grounded encoder — live endpoint
POST a PNG → get fsaverage5 cortical activations + region scores
(FFA, V4, MT+, PFC, ACC, Insula) + subcortical proxies.

Endpoint: https://b227-34-143-253-73.ngrok-free.app/encode
Health:   https://b227-34-143-253-73.ngrok-free.app/health

Python:
import requests
with open('thumb.png','rb') as f:
    r = requests.post('https://b227-34-143-253-73.ngrok-free.app/encode', data=f.read(), timeout=120)
print(r.json()['region_scores'])

curl:
curl -X POST --data-binary @thumb.png https://b227-34-143-253-73.ngrok-free.app/encode

Notes:
- Hosted on my Colab — ping me if it 502s
- ~40 req/min cap (ngrok free), ~20s first call (warm-up)
- Subcortical = cortical-proxy estimates, not raw subcortical signal



In [2]:
from google.colab import files
import requests, json

URL = "https://0272-34-178-112-199.ngrok-free.app/encode"

up = files.upload()
fname = next(iter(up))
print(f"\nSending {fname} ({len(up[fname])} bytes)...\n")

r = requests.post(URL, data=up[fname], timeout=180)
print(f"Status: {r.status_code}\n")

if r.status_code == 200:
    out = r.json()
    print("Cortical region scores:")
    for region, score in out['region_scores'].items():
        bar = "█" * int(score * 30)
        print(f"  {region:8s} {score:.3f}  {bar}")
    print(f"\nSubcortical (proxy): {out['subcortical_estimates']['values']}")
    print(f"Vertices: {len(out['vertices']['lh'])} per hemi")
    print(f"Timesteps: {out['meta']['n_timesteps']}")
else:
    print(r.json())

StopIteration: 

In [3]:
import cv2, io, os, tempfile
import numpy as np
import pandas as pd
from PIL import Image
from scipy.spatial import cKDTree
from nilearn.surface import load_surf_mesh
from tribev2.demo_utils import get_audio_and_text_events

def tribe_predict_image(img_bytes: bytes) -> dict:
    img    = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_np = np.array(img)
    h, w   = img_np.shape[:2]
    w, h   = w - w % 2, h - h % 2
    img_np = img_np[:h, :w]

    with tempfile.TemporaryDirectory() as tmpdir:
        video_path = os.path.join(tmpdir, 'ui.mp4')
        writer = cv2.VideoWriter(
            video_path, cv2.VideoWriter_fourcc(*'mp4v'), 1, (w, h))
        for _ in range(2):
            writer.write(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
        writer.release()
        event_df = pd.DataFrame([{
            "type":"Video","filepath":video_path,
            "start":0,"timeline":"default","subject":"default"}])
        events   = get_audio_and_text_events(event_df)
        preds, _ = model.predict(events=events)

    preds_np = np.asarray(preds)
    mean_act = preds_np.mean(axis=0)

    scores = {r: float(mean_act[m].mean()) if m.sum() > 0 else 0.0
              for r, m in region_masks.items()}
    lo, hi = min(scores.values()), max(scores.values())
    if hi - lo > 1e-6:
        scores = {k: (v-lo)/(hi-lo) for k,v in scores.items()}

    lh_coords = load_surf_mesh(fsavg['pial_left'])[0]
    rh_coords = load_surf_mesh(fsavg['pial_right'])[0]
    all_coords = np.vstack([lh_coords, rh_coords])
    tree = cKDTree(all_coords)
    subcortical_scores = {}
    for region, mni_coords in subcortical_masks.items():
        _, idx = tree.query(mni_coords)
        subcortical_scores[region] = float(mean_act[idx].mean())
    lo2, hi2 = min(subcortical_scores.values()), max(subcortical_scores.values())
    if hi2 - lo2 > 1e-6:
        subcortical_scores = {k:(v-lo2)/(hi2-lo2) for k,v in subcortical_scores.items()}

    return {
        "vertices":{"lh":mean_act[:10242].tolist(),"rh":mean_act[10242:].tolist()},
        "region_scores": scores,
        "subcortical_estimates":{"values":subcortical_scores,
            "method":"nearest_cortical_vertex",
            "warning":"Cortical proxy estimates only. TRIBE v2 models cortical activity only."},
        "meta":{"n_vertices_per_hemi":10242,
                "n_timesteps":int(preds_np.shape[0]),
                "surface":"fsaverage5"}}

print("✓ tribe_predict_image ready")
print("✓ all deps:", all(n in globals() for n in
                         ['model','region_masks','subcortical_masks','fsavg']))

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-20 12:22:55 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


✓ tribe_predict_image ready
✓ all deps: False


In [4]:
# === FULL STATE REBUILD — one cell, ~90s ===
import os, urllib.request, cv2, io, tempfile
from pathlib import Path
from google.colab import userdata

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
from scipy.spatial import cKDTree
from nilearn import datasets as nl_datasets
from nilearn.surface import load_surf_mesh
from tribev2.demo_utils import TribeModel, get_audio_and_text_events

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 1. Model
CACHE_DIR = Path('/content/tribe_cache'); CACHE_DIR.mkdir(exist_ok=True)
print('[1/5] Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=str(CACHE_DIR))

# 2. fsaverage5
print('[2/5] Fetching fsaverage5...')
fsavg = nl_datasets.fetch_surf_fsaverage(mesh='fsaverage5')

# 3. Cortical masks
print('[3/5] Building cortical region masks...')
atlas_path = '/content/tribe_cache/MMP_in_MNI_corr.nii.gz'
if not os.path.exists(atlas_path):
    urllib.request.urlretrieve('https://ndownloader.figshare.com/files/5594363', atlas_path)
atlas_img  = nib.load(atlas_path)
atlas_data = np.asarray(atlas_img.get_fdata(), dtype=np.int32)
inv_affine = np.linalg.inv(atlas_img.affine)
lh_c = load_surf_mesh(fsavg['pial_left'])[0]
rh_c = load_surf_mesh(fsavg['pial_right'])[0]
all_c = np.vstack([lh_c, rh_c])
vox = np.round((inv_affine @ np.hstack([all_c, np.ones((len(all_c),1))]).T).T[:, :3]).astype(int)
nx,ny,nz = atlas_data.shape
ok = ((vox[:,0]>=0)&(vox[:,0]<nx)&(vox[:,1]>=0)&(vox[:,1]<ny)&(vox[:,2]>=0)&(vox[:,2]<nz))
vlabels = np.zeros(len(all_c), dtype=np.int32)
vi = vox[ok]; vlabels[ok] = atlas_data[vi[:,0],vi[:,1],vi[:,2]]
GLASSER = {"V4":6,"FFC":18,"MT":23,"MST":2,"V4t":156,"FST":157,"a24":61,"p24":179,
           "p32":63,"s32":64,"24dd":40,"24dv":41,"46":84,"9-46d":86,"p9-46v":83,
           "a9-46v":85,"AAIC":112,"AVI":111,"MI":109,"PI":110}
REGION_MAP = {"FFA":("FFC",),"V4":("V4",),"MT+":("MT","MST","V4t","FST"),
              "PFC":("46","9-46d","p9-46v","a9-46v"),
              "ACC":("a24","p24","p32","s32","24dd","24dv"),
              "Insula":("AAIC","AVI","MI","PI")}
region_masks = {}
for region, names in REGION_MAP.items():
    idx = [i for n in names for lh in [GLASSER.get(n)] if lh for i in (lh, lh+180)]
    if idx: region_masks[region] = np.isin(vlabels, idx)

# 4. Subcortical masks
print('[4/5] Building subcortical masks...')
atlas2 = nl_datasets.fetch_atlas_harvard_oxford('sub-maxprob-thr25-2mm')
sub_img = atlas2.maps if hasattr(atlas2.maps,'get_fdata') else nib.load(atlas2.maps)
sub_data = np.asarray(sub_img.get_fdata(), dtype=np.int32)
SUBLBL = {'Hippocampus':[6,13],'Amygdala':[7,14],'NAcc':[8,15]}
subcortical_masks = {r: nib.affines.apply_affine(sub_img.affine, np.argwhere(np.isin(sub_data, ids)))
                     for r, ids in SUBLBL.items()}

# 5. Predict function
print('[5/5] Defining tribe_predict_image...')
def tribe_predict_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_np = np.array(img); h,w = img_np.shape[:2]
    w, h = w - w%2, h - h%2; img_np = img_np[:h,:w]
    with tempfile.TemporaryDirectory() as td:
        vp = os.path.join(td,'ui.mp4')
        wr = cv2.VideoWriter(vp, cv2.VideoWriter_fourcc(*'mp4v'), 1, (w,h))
        for _ in range(2): wr.write(cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
        wr.release()
        ev = pd.DataFrame([{"type":"Video","filepath":vp,"start":0,
                            "timeline":"default","subject":"default"}])
        events = get_audio_and_text_events(ev)
        preds, _ = model.predict(events=events)
    p = np.asarray(preds); m = p.mean(axis=0)
    s = {r: float(m[mk].mean()) if mk.sum()>0 else 0.0 for r,mk in region_masks.items()}
    lo,hi = min(s.values()), max(s.values())
    if hi-lo > 1e-6: s = {k:(v-lo)/(hi-lo) for k,v in s.items()}
    tree = cKDTree(all_c)
    ss = {}
    for r, mc in subcortical_masks.items():
        _, ix = tree.query(mc); ss[r] = float(m[ix].mean())
    lo2,hi2 = min(ss.values()), max(ss.values())
    if hi2-lo2 > 1e-6: ss = {k:(v-lo2)/(hi2-lo2) for k,v in ss.items()}
    return {"vertices":{"lh":m[:10242].tolist(),"rh":m[10242:].tolist()},
            "region_scores":s,
            "subcortical_estimates":{"values":ss,"method":"nearest_cortical_vertex",
                "warning":"Cortical proxy estimates only."},
            "meta":{"n_vertices_per_hemi":10242,"n_timesteps":int(p.shape[0]),
                    "surface":"fsaverage5"}}

print('\n' + '='*50)
print('STATE CHECK:')
for n in ['model','fsavg','region_masks','subcortical_masks','tribe_predict_image']:
    print(f"  {'✓' if n in globals() else '✗'} {n}")
print('='*50)
print('Ready. Run your upload+POST cell next.')

[1/5] Loading TRIBE v2...


/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-20 12:22:57 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('

[2/5] Fetching fsaverage5...
[3/5] Building cortical region masks...
[4/5] Building subcortical masks...


[fetch_atlas_harvard_oxford] Dataset found in /root/nilearn_data/fsl

[5/5] Defining tribe_predict_image...

STATE CHECK:
  ✓ model
  ✓ fsavg
  ✓ region_masks
  ✓ subcortical_masks
  ✓ tribe_predict_image
Ready. Run your upload+POST cell next.


In [5]:
from google.colab import files
import requests

URL = "https://0272-34-178-112-199.ngrok-free.app/encode"
up = files.upload()
fname = next(iter(up))
print(f"\nSending {fname} ({len(up[fname])} bytes)...\n")

r = requests.post(URL, data=up[fname], timeout=180)
print(f"Status: {r.status_code}\n")

if r.status_code == 200:
    out = r.json()
    print("Cortical region scores:")
    for region, score in out['region_scores'].items():
        bar = "█" * int(score * 30)
        print(f"  {region:8s} {score:.3f}  {bar}")
    print(f"\nSubcortical (proxy): {out['subcortical_estimates']['values']}")
    print(f"Timesteps: {out['meta']['n_timesteps']}")
else:
    print(r.json())

StopIteration: 

In [6]:
print(r.json())

NameError: name 'r' is not defined

In [7]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("arch list:", torch.cuda.get_arch_list())
print("device cap:", torch.cuda.get_device_capability(0))

torch: 2.6.0+cu124
cuda: 12.4
arch list: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
device cap: (8, 0)


In [8]:
from google.colab import files
up = files.upload()
import requests
fname = next(iter(up))
r = requests.post("https://35cd-8-228-24-248.ngrok-free.app/encode",
                  data=up[fname], timeout=180)
print(r.status_code)
print(r.json() if r.status_code != 200 else list(r.json().keys()))

StopIteration: 

In [9]:
print("cuda:",torc.version.cuda)
print("arch list:",torch.cuda.get_arch_list())

NameError: name 'torc' is not defined